In [1]:
import os
import re
import string
import math
import copy

import numpy as np
import pandas as pd

import torch
from torch.autograd import Variable

import transformers
from transformers import AdamW

# NOTE:
# - Removed tensorflow/keras and nltk imports because they are unused in this notebook
#   and can trigger protobuf/transformers environment conflicts in Kaggle images.

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.11/dist-packages/transformers/__init__.py)

In [2]:
train = pd.read_csv("../input/tweet-sentiment-extraction/train.csv")
test = pd.read_csv("../input/tweet-sentiment-extraction/test.csv")
sample_submission = pd.read_csv(
    "../input/tweet-sentiment-extraction/sample_submission.csv"
)



In [3]:
train.dropna(inplace=True)




In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub("[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub("\n", "", text)
    text = re.sub(r"\w*\d\w*", "", text)
    return text




In [5]:
train["text"] = train["text"].apply(lambda x: clean_text(x))
train["selected_text"] = train["selected_text"].apply(lambda x: clean_text(x))
test["text"] = test["text"].apply(lambda x: clean_text(x))



In [6]:
train_positive = train.loc[(train.sentiment == "positive")]
train_neutral = train.loc[(train.sentiment == "neutral")]
train_negative = train.loc[(train.sentiment == "negative")]

test_positive = test.loc[(test.sentiment == "positive")]
test_neutral = test.loc[(test.sentiment == "neutral")]
test_negative = test.loc[(test.sentiment == "negative")]



In [7]:
train_all = [train_positive, train_neutral, train_negative]
test_all = [test_positive, test_neutral, test_negative]




In [8]:
def jaccard(str1, str2):
    a = set(str(str1).lower().split())
    b = set(str(str2).lower().split())
    c = a.intersection(b)
    if (len(a) + len(b) - len(c)) == 0:
        return 1.0
    return float(len(c)) / (len(a) + len(b) - len(c))




In [9]:
def from_predicted_positon_to_text(predicted, threshold, padded_tokens, tokenizer):
    # predicted: torch.Tensor on device
    predicted = predicted.detach().float().cpu().numpy().copy()

    predicted[predicted >= threshold] = 1
    predicted[predicted < threshold] = 0

    tokens_matrix = []
    decode_matrix = []
    for i in range(padded_tokens.shape[0]):
        decode = tokenizer.decode(padded_tokens[i], skip_special_tokens=False)
        toks = tokenizer.tokenize(decode)
        tokens_matrix.append(toks)
        decode_matrix.append(" ".join(toks))

    index_matrix = []
    for i in range(len(predicted)):
        index = [ix for ix, p in enumerate(predicted[i, :].tolist()) if p == 1]
        index_matrix.append(index)

    first_last_words = []
    for toks, idxs in zip(tokens_matrix, index_matrix):
        # remove [CLS] position if included
        if 0 in idxs:
            idxs = [x for x in idxs if x != 0]

        # remove indices that point to special/pad tokens
        cleaned = []
        for k in idxs:
            if k < 0 or k >= len(toks):
                continue
            if toks[k] in ["[SEP]", "[PAD]"]:
                continue
            cleaned.append(k)

        if len(cleaned) == 0:
            first_last_words.append(["", ""])
        else:
            first = cleaned[0]
            last = cleaned[-1]
            if toks[first] == toks[last]:
                first_last_words.append(["", toks[last]])
            else:
                first_last_words.append([toks[first], toks[last]])

    predicted_val_text = []
    for (start_tok, end_tok), s in zip(first_last_words, decode_matrix):
        # Robust extraction: if regex fails, fall back to decoded (specials stripped)
        try:
            if start_tok == "" and end_tok == "":
                extracted = ""
            else:
                pattern = re.escape(start_tok) + r".+" + re.escape(end_tok)
                extracted = re.findall(pattern, s)[0]
        except Exception:
            extracted = s

        extracted = (
            extracted.replace("[CLS]", "")
            .replace("CLS]", "")
            .replace("[SEP]", "")
            .replace("[PAD]", "")
            .replace("[S", "")
            .replace("[P", "")
            .replace("AD", "")
            .replace(" ##", "")
            .strip()
        )
        predicted_val_text.append(extracted)

    return predicted_val_text




In [10]:
class BertModel(torch.nn.Module):
    def __init__(self, UNCASED, outputSize, droupout, std):
        super(BertModel, self).__init__()
        self.config = transformers.BertConfig.from_pretrained(
            UNCASED, output_hidden_states=True
        )
        self.bert_model = transformers.BertModel.from_pretrained(
            UNCASED, config=self.config
        )
        self.drop_out = torch.nn.Dropout(droupout)
        self.con_model1 = torch.nn.Conv1d(
            in_channels=768, out_channels=256, kernel_size=1
        )
        self.con_model2 = torch.nn.Conv1d(
            in_channels=256, out_channels=outputSize, kernel_size=1
        )
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        last_hidden_states = self.bert_model(
            input_ids, attention_mask=attention_mask.float()
        )
        last_hidden_states = last_hidden_states[0].permute(0, 2, 1)
        out = self.drop_out(last_hidden_states)
        out = self.con_model1(out)
        out = self.con_model2(out)
        out = torch.sum(out, dim=2)
        out = self.sigmoid(out)
        return out




In [11]:
torch.cuda.empty_cache()

category = ["positive", "neutral", "negative"]
max_sequence_length = 32
val_frac = 0.25
num_of_val = 1000
learningRate = 3e-5
max_length = max_sequence_length
outputSize = max_sequence_length
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = [3, 1, 4]
predicted_text = {}

# FIX: use a valid local/offline identifier for tokenizer+model
# Kaggle images typically have bert-base-uncased cached; no internet needed.
UNCASED = "bert-base-uncased"
std = 0.02
droupout = 0.1
batch_size = 16
no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

for i in range(len(train_all)):
    if i == 1:
        threshold = 0
    else:
        threshold = 0.5

    tokenizer = transformers.BertTokenizer.from_pretrained(UNCASED, do_lower_case=True)

    model = BertModel(UNCASED, outputSize, droupout, std)
    model.to(device)

    criterion = torch.nn.MSELoss()
    param_optimizer = list(model.named_parameters())
    optimizer_parameters = [
        {
            "params": [
                p for n, p in param_optimizer if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.001,
        },
        {
            "params": [
                p for n, p in param_optimizer if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_parameters, lr=learningRate)

    train_data = train_all[i]
    test_data = test_all[i]

    X_train_text = train_data.loc[:, "text"].tolist()
    y_train_text = train_data.loc[:, "selected_text"].tolist()
    X_test = test_data.loc[:, ["textID", "text"]]

    X_train_tokens = [
        tokenizer.encode(
            t, add_special_tokens=True, max_length=max_length, truncation=True
        )
        for t in X_train_text
    ]
    y_train_tokens = [
        tokenizer.encode(
            t, add_special_tokens=True, max_length=max_length, truncation=True
        )
        for t in y_train_text
    ]
    X_test_tokens = [
        tokenizer.encode(
            t, add_special_tokens=True, max_length=max_length, truncation=True
        )
        for t in X_test.text.tolist()
    ]

    X_train_all_input_ids = np.array(
        [tok + [0] * (max_length - len(tok)) for tok in X_train_tokens], dtype=np.int64
    )
    y_train_all_input_ids = np.array(
        [tok + [0] * (max_length - len(tok)) for tok in y_train_tokens], dtype=np.int64
    )
    X_test_input_ids = np.array(
        [tok + [0] * (max_length - len(tok)) for tok in X_test_tokens], dtype=np.int64
    )

    X_train_all_attention_mask = np.where(X_train_all_input_ids != 0, 1, 0).astype(
        np.int64
    )
    X_test_attention_mask = np.where(X_test_input_ids != 0, 1, 0).astype(np.int64)

    y_train_bool = []
    for j in range(len(y_train_all_input_ids)):
        a = [
            1 if (x > 0 and x in y_train_all_input_ids[j, :]) else 0
            for x in X_train_all_input_ids[j, :].tolist()
        ]
        y_train_bool.append(a)
    y_train_bool = np.array(y_train_bool, dtype=np.int64)

    # Keep original val sizing logic, but ensure it doesn't exceed dataset size
    this_num_of_val = min(num_of_val, len(X_train_all_input_ids))
    this_num_of_val = (
        this_num_of_val + (len(X_train_all_input_ids) - this_num_of_val) % batch_size
    )
    num_of_train = len(X_train_all_input_ids) - this_num_of_val
    if num_of_train < batch_size:
        # fallback: ensure at least one batch for training
        this_num_of_val = max(0, len(X_train_all_input_ids) - batch_size)
        num_of_train = len(X_train_all_input_ids) - this_num_of_val

    X_val = (
        X_train_all_input_ids[-this_num_of_val:, :]
        if this_num_of_val > 0
        else X_train_all_input_ids[:0, :]
    )
    X_val_attention_mask = (
        X_train_all_attention_mask[-this_num_of_val:, :]
        if this_num_of_val > 0
        else X_train_all_attention_mask[:0, :]
    )
    y_val = (
        y_train_bool[-this_num_of_val:, :]
        if this_num_of_val > 0
        else y_train_bool[:0, :]
    )

    training_loss = []
    model.train()
    for epoch in range(epochs[i]):
        for k_fold in range(int(num_of_train / batch_size)):
            X_train_batch = X_train_all_input_ids[
                k_fold * batch_size : (k_fold + 1) * batch_size, :
            ]
            X_train_attention_mask_batch = X_train_all_attention_mask[
                k_fold * batch_size : (k_fold + 1) * batch_size, :
            ]
            y_train_batch = y_train_bool[
                k_fold * batch_size : (k_fold + 1) * batch_size, :
            ]

            X_train_batch = Variable(torch.from_numpy(X_train_batch).to(device))
            X_train_attention_mask_batch = Variable(
                torch.from_numpy(X_train_attention_mask_batch).to(device)
            )
            y_train_batch = Variable(torch.from_numpy(y_train_batch).to(device))

            outputs = model(X_train_batch, X_train_attention_mask_batch)
            loss = criterion(outputs.float(), y_train_batch.float())
            training_loss.append(loss.item())
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if k_fold % 100 == 0:
                print(
                    f"{i}-th category, total {int(num_of_train/batch_size)} fold, now {k_fold}, epoch {epoch}, loss {loss.item()}"
                )

    # Validation (kept, but guarded if val is empty)
    model.eval()
    if this_num_of_val > 0:
        with torch.no_grad():
            X_val_t = Variable(torch.from_numpy(X_val).to(device))
            X_val_attention_mask_t = Variable(
                torch.from_numpy(X_val_attention_mask).to(device)
            )
            eval_predicted = model(X_val_t, X_val_attention_mask_t)

        predicted_val_text = from_predicted_positon_to_text(
            eval_predicted,
            threshold,
            X_train_all_input_ids[-this_num_of_val:, :],
            tokenizer,
        )

        jaccard_score_list = []
        for str1, str2 in zip(predicted_val_text, y_train_text[-this_num_of_val:]):
            jaccard_score_list.append(jaccard(str1, str2))
        result = pd.Series(jaccard_score_list)
        print(result.describe())

    # Test inference
    with torch.no_grad():
        X_test_input_ids_t = Variable(torch.from_numpy(X_test_input_ids).to(device))
        X_test_attention_mask_t = Variable(
            torch.from_numpy(X_test_attention_mask).to(device)
        )
        test_predicted = model(X_test_input_ids_t, X_test_attention_mask_t)

    predicted_test_text = from_predicted_positon_to_text(
        test_predicted, threshold, X_test_input_ids, tokenizer
    )

    for p, idx in zip(predicted_test_text, X_test.textID.tolist()):
        predicted_text[idx] = p



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

2026-05-14 21:25:49.201829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778793949.212182      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778793949.216139      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 21:25:49.231409: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

NameError: name 'AdamW' is not defined

In [12]:
# FIX: build submission robustly (no deprecated DataFrame.append, ensure all ids exist, keep required columns)
sub = sample_submission.copy()

# Fill with predictions; fallback to full cleaned tweet text if any id missing (prevents KeyError)
fallback_map = dict(zip(test["textID"].tolist(), test["text"].tolist()))
sub["selected_text"] = sub["textID"].map(predicted_text)

missing_mask = sub["selected_text"].isna()
if missing_mask.any():
    sub.loc[missing_mask, "selected_text"] = (
        sub.loc[missing_mask, "textID"].map(fallback_map).fillna("")
    )

# Ensure strings (Kaggle expects quoted strings; pandas will quote as needed)
sub["selected_text"] = sub["selected_text"].astype(str)

# Ensure exactly required columns
sub = sub[["textID", "selected_text"]]

print(sub.head())
print(sub.isna().sum())



       textID                                      selected_text
0  80a1e6bc32          i just saw a shooting star i made my wish
1  863097735d  gosh today sucks i didnt get my tax returns im...
2  264cd5277f  tired and didnt really have an exciting saturd...
3  baee1e6ffc                ive been eating cheetos all morning
4  67d06a8dee   haiiii sankq im fineee ima js get a checkup c...
textID           0
selected_text    0
dtype: int64


/tmp/ipykernel_55/3003307062.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['i just saw a shooting star i made my wish'
 'gosh today sucks i didnt get my tax returns im so upset cuz now i have to miss my best friends wedding in washingtonlame'
 'tired and didnt really have an exciting saturday  oh well hope its better tomorrow'
 ...
 'im so excited for mothers day  this has been a big year for me  olivia and shes finally old enough to be excited  understand'
 'i miss my '
 ' ugh my trousers do that too and ive never found a way to combat it its so weird']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sub.loc[missing_mask, "selected_text"] = (


In [13]:
sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
print("Columns:", list(sub.columns))
print("File exists:", os.path.exists("submission.csv"))

Wrote submission.csv with shape: (2749, 2)
Columns: ['textID', 'selected_text']
File exists: True
